# AML model comparison, tuning and ablation
Runs on a **CPU** Colab runtime (~45 min). Uses the duration and word-gap tables written by section 3 of
`phonetic_guided_tts.ipynb` (`dur_full.csv`, `pause_gaps.csv`); set `DATA_DIR` to where they are.
Results are written to `outputs/results/aml_*.csv`.

1. Load tables · 2. Duration models (per-phoneme mean, Ridge, random forest, MLP, LightGBM) with paired bootstrap CIs ·
3. LightGBM tuning (3-fold CV by sentence), feature ablation, error by sound class · 4. Pause models (rule, logistic
regression, random forest, MLP, LightGBM)

## 1. Load tables

In [ ]:
import os, time, numpy as np, pandas as pd
# Tables written by phonetic_guided_tts.ipynb (section 3). Point DATA_DIR at wherever they are.
DATA_DIR = "/content/work"                    # e.g. "/content/drive/MyDrive" if you saved them to Drive
OUT = "/content/outputs/results"; os.makedirs(OUT, exist_ok=True)
RAW = "https://raw.githubusercontent.com/shiavm006/phonetic-guided-tts/main/results"

dur = pd.read_csv(f"{DATA_DIR}/dur_full.csv", keep_default_na=False)
split = pd.read_csv(f"{RAW}/split.csv")
d = dur[dur.edge_sil == 0].merge(split, on="utt")
tr = d[d.split == "train"].reset_index(drop=True)
te = d[d.split == "test"].reset_index(drop=True)
MS = 1000 * 256 / 22050                                   # one mel frame in ms
CAT = ["phone", "prev", "next"]
NUM = ["stress", "pos_in_word", "word_len", "before_pause", "rel_pos", "n_phones"]
print(f"train {len(tr):,} phonemes | test {len(te):,} phonemes")

## 2. Duration models

In [ ]:
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

onehot = lambda: ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), CAT), ("n", StandardScaler(), NUM)])
ordinal = lambda: ColumnTransformer([("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT), ("n", "passthrough", NUM)])
CATS = {c: sorted(d[c].unique()) for c in CAT}
def lgbX(x, cols=CAT + NUM):
    X = x[cols].copy()
    for c in cols:
        if c in CATS: X[c] = pd.Categorical(X[c], categories=CATS[c])
    return X
sub = tr.sample(200_000, random_state=0)                  # RF and MLP train on a 200k subsample (CPU time)

preds, info = {}, {}
preds["Per-phoneme mean"] = te.phone.map(tr.groupby("phone").frames.mean()).fillna(tr.frames.mean()).values
info["Per-phoneme mean"] = (len(tr), 0)
for name, model, data in [
        ("Ridge (linear)", make_pipeline(onehot(), Ridge(alpha=1.0)), tr),
        ("Random forest", make_pipeline(ordinal(), RandomForestRegressor(n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0)), sub),
        ("MLP (128-64)", make_pipeline(onehot(), MLPRegressor((128, 64), early_stopping=True, max_iter=40, random_state=0)), sub)]:
    t0 = time.time(); model.fit(data[CAT + NUM], data.frames)
    preds[name] = model.predict(te[CAT + NUM]); info[name] = (len(data), time.time() - t0); print("done:", name)
t0 = time.time()
lgbm = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=63, verbose=-1).fit(lgbX(tr), tr.frames)
preds["LightGBM"] = lgbm.predict(lgbX(te)); info["LightGBM"] = (len(tr), time.time() - t0)

codes, uniq = pd.factorize(te.utt); cnt = np.bincount(codes)
usum = lambda e: np.bincount(codes, weights=e)
idx = np.random.default_rng(0).integers(0, len(uniq), (2000, len(uniq)))
lg_s = usum(np.abs(preds["LightGBM"] - te.frames))
sil = (te.phone == "sil").values
rows = []
for name, p in preds.items():
    e = np.abs(p - te.frames.values); s = usum(e)
    diff = (s[idx].sum(1) - lg_s[idx].sum(1)) / cnt[idx].sum(1) * MS
    rows.append(dict(model=name, mae_all_ms=e.mean() * MS, mae_speech_ms=e[~sil].mean() * MS, mae_pause_ms=e[sil].mean() * MS,
                     vs_lightgbm_ci=f"[{np.percentile(diff, 2.5):+.1f}, {np.percentile(diff, 97.5):+.1f}]",
                     train_rows=info[name][0], train_sec=round(info[name][1])))
dur_models = pd.DataFrame(rows); dur_models.to_csv(f"{OUT}/aml_duration_models.csv", index=False)
print(dur_models.round(1).to_string(index=False))

## 3. Tuning, ablation and error by sound class

In [ ]:
from sklearn.model_selection import GroupKFold
grid = [dict(num_leaves=31, learning_rate=0.05), dict(num_leaves=63, learning_rate=0.05),
        dict(num_leaves=127, learning_rate=0.05), dict(num_leaves=63, learning_rate=0.1)]
tune = []
for g in grid:
    maes = []
    for a, b in GroupKFold(3).split(sub, groups=sub.utt):
        m = lgb.LGBMRegressor(n_estimators=400, verbose=-1, **g).fit(lgbX(sub.iloc[a]), sub.frames.iloc[a])
        maes.append(np.abs(m.predict(lgbX(sub.iloc[b])) - sub.frames.iloc[b]).mean() * MS)
    tune.append(dict(**g, cv_mae_ms=np.mean(maes), cv_std=np.std(maes)))
tune = pd.DataFrame(tune); tune.to_csv(f"{OUT}/aml_tuning.csv", index=False)

groups = {"phone identity": ["phone"], "neighbours": ["prev", "next"], "stress": ["stress"],
          "position": ["pos_in_word", "word_len", "rel_pos", "n_phones"], "following pause": ["before_pause"]}
abl = [dict(removed="nothing (full model)", mae_all_ms=np.abs(preds["LightGBM"] - te.frames).mean() * MS,
            mae_speech_ms=np.abs(preds["LightGBM"] - te.frames)[~sil].mean() * MS)]
for gname, cols in groups.items():
    keep = [c for c in CAT + NUM if c not in cols]
    m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=63, verbose=-1).fit(lgbX(tr, keep), tr.frames)
    e = np.abs(m.predict(lgbX(te, keep)) - te.frames)
    abl.append(dict(removed=gname, mae_all_ms=e.mean() * MS, mae_speech_ms=e[~sil].mean() * MS))
abl = pd.DataFrame(abl); abl["increase_ms"] = abl.mae_all_ms - abl.mae_all_ms.iloc[0]
abl.to_csv(f"{OUT}/aml_ablation.csv", index=False)

CLASS = {**{p: "vowel" for p in "AA AE AH AO AW AY EH ER EY IH IY OW OY UH UW".split()},
         **{p: "stop" for p in "B D G K P T".split()}, **{p: "fricative" for p in "F V TH DH S Z SH ZH HH".split()},
         **{p: "affricate" for p in "CH JH".split()}, **{p: "nasal" for p in "M N NG".split()},
         **{p: "liquid/glide" for p in "L R W Y".split()}, "sil": "pause", "spn": "unknown word"}
cls = te.phone.map(CLASS).fillna("other")
byc = pd.DataFrame({"class": cls, "n": 1, "true_ms": te.frames * MS,
                    "phone_mean": np.abs(preds["Per-phoneme mean"] - te.frames) * MS,
                    "lightgbm": np.abs(preds["LightGBM"] - te.frames) * MS}).groupby("class").agg(
    n=("n", "sum"), mean_duration_ms=("true_ms", "mean"), mae_phone_mean=("phone_mean", "mean"), mae_lightgbm=("lightgbm", "mean"))
byc.to_csv(f"{OUT}/aml_error_by_class.csv")
print("TUNING\n", tune.round(2).to_string(index=False))
print("\nABLATION\n", abl.round(1).to_string(index=False))
print("\nBY CLASS\n", byc.round(1).sort_values("n", ascending=False).to_string())

## 4. Pause placement models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_recall_fscore_support

g = pd.read_csv(f"{DATA_DIR}/pause_gaps.csv", keep_default_na=False)
if "split" not in g: g = g.merge(split, on="utt")
trg = g[g.split == "train"]
top = set(trg.cur_w.value_counts().head(80).index) | set(trg.next_w.value_counts().head(80).index)
for c in ["cur_w", "next_w"]: g[c + "_c"] = g[c].where(g[c].isin(top), "<other>")
GC = ["punct", "cur_w_c", "next_w_c"]
GN = ["wlen", "next_wlen", "rel_pos", "n_words", "to_end", "since_punct", "since_chars"]
trg = g[g.split == "train"]; teg = g[g.split == "test"]
dev_u = set(np.random.default_rng(0).choice(trg.utt.unique(), trg.utt.nunique() // 10, replace=False))
fit_g, dev_g = trg[~trg.utt.isin(dev_u)], trg[trg.utt.isin(dev_u)]
GCATS = {c: sorted(g[c].unique()) for c in GC}
def gX(x):
    X = x[GC + GN].copy()
    for c in GC: X[c] = pd.Categorical(X[c], categories=GCATS[c])
    return X
oh = lambda: ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), GC), ("n", StandardScaler(), GN)])
od = lambda: ColumnTransformer([("c", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), GC), ("n", "passthrough", GN)])
models = {"Logistic regression": (make_pipeline(oh(), LogisticRegression(max_iter=1000)), lambda x: x[GC + GN]),
          "Random forest": (make_pipeline(od(), RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0)), lambda x: x[GC + GN]),
          "MLP (64-32)": (make_pipeline(oh(), MLPClassifier((64, 32), early_stopping=True, max_iter=60, random_state=0)), lambda x: x[GC + GN]),
          "LightGBM": (lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63, verbose=-1), gX)}
def score(name, p):
    pr, rc, f1, _ = precision_recall_fscore_support(teg.pause, p, average="binary", zero_division=0)
    return dict(model=name, precision=pr * 100, recall=rc * 100, f1=f1 * 100)
rows = [score("Rule: pause iff punctuation", (teg.punct != "none").astype(int))]
for name, (m, X) in models.items():
    m.fit(X(fit_g), fit_g.pause)
    pdev = m.predict_proba(X(dev_g))[:, 1]
    th = max(np.arange(0.2, 0.71, 0.05), key=lambda t: precision_recall_fscore_support(dev_g.pause, pdev > t, average="binary", zero_division=0)[2])
    rows.append({**score(name, (m.predict_proba(X(teg))[:, 1] > th).astype(int)), "threshold": round(th, 2)})
pause_models = pd.DataFrame(rows); pause_models.to_csv(f"{OUT}/aml_pause_models.csv", index=False)
print(pause_models.round(1).to_string(index=False))